# Dana's Bakery Order Processing System

## 1. Project Introduction

A deterministic order workflow built with Python and LangGraph. It validates orders, checks stock, tries limited substitutions, calculates discounts and fulfillment, and generates a friendly response. No LLM, API key, or paid service is used. Each call starts with the original inventory.


## 2. Environment Setup

Run in Google Colab to install LangGraph.


In [ ]:
%pip -q install langgraph


## 3. Business Rules

Menu and stock below apply independently to every order. Croissants are discontinued; orders must be fulfilled in full. Substitution order is babka → rugelach → sourdough → challah, up to two alternatives. A 15% discount applies only above 150 ILS and not Friday. Jerusalem delivery costs 20 ILS; elsewhere is pickup. Decimal arithmetic avoids float errors. Only positive integer quantities are allowed (including rejection of booleans and numeric strings). Empty city means pickup. Inventory is not persisted. Messages use deterministic templates. Accounts, payments, Instagram, delivery integrations, permanent inventory storage, and LLM messages are out of scope.


In [ ]:
from decimal import Decimal
MENU_PRICES = {"challah": Decimal("18"), "babka": Decimal("45"), "rugelach": Decimal("30"), "sourdough": Decimal("25")}
INITIAL_INVENTORY = {"challah": 10, "babka": 0, "rugelach": 3, "sourdough": 4}
DISCONTINUED = {"croissant"}
SEQUENCE = ["babka", "rugelach", "sourdough", "challah"]
MAX_ALTERNATIVES = 2
WEEKDAYS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}


## 4. State Definition

The state contains input, working values, required output, and an additive step-history reducer.


In [ ]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END

class BakeryState(TypedDict, total=False):
    item: object
    quantity: object
    day: object
    city: object
    original_item: object
    normalized_day: str
    normalized_city: str
    alternative_attempts: int
    stock_sufficient: bool
    subtotal: Decimal
    discount: Decimal
    delivery_fee: Decimal
    rejection_reason: str
    status: str
    delivery: str
    total: Decimal
    message: str
    steps: Annotated[list[str], operator.add]


## 5. Graph Nodes

Each node handles one task and adds its name to the reducer-backed step history.


In [ ]:
def validate_order(s: BakeryState) -> dict:
    raw_item, qty, raw_day, raw_city = (s.get(k) for k in ("item", "quantity", "day", "city"))
    u = {"steps": ["validate_order"], "original_item": raw_item, "alternative_attempts": 0,
         "subtotal": Decimal(0), "discount": Decimal(0), "delivery_fee": Decimal(0),
         "total": Decimal(0), "delivery": "pickup"}
    errors = []
    if not isinstance(raw_item, str) or not raw_item.strip():
        item = raw_item
        errors.append("Please provide a product from our active menu.")
    else:
        item = " ".join(raw_item.split()).casefold()
        u["item"] = item
        u["original_item"] = item
        if item in DISCONTINUED: errors.append("Croissants are discontinued and are not on our active menu.")
        elif item not in MENU_PRICES: errors.append(f"{item.capitalize()} is not on our active menu.")
    if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
        errors.append("Quantity must be a positive whole number.")
    if not isinstance(raw_day, str) or not raw_day.strip():
        day = ""
        errors.append("Please provide a valid day of the week.")
    else:
        day = " ".join(raw_day.split()).casefold()
        if day not in WEEKDAYS: errors.append("Please provide a valid day of the week.")
    u["normalized_day"] = day
    if not isinstance(raw_city, str):
        city = ""
        errors.append("Please provide a city as text (or an empty city for pickup).")
    else: city = " ".join(raw_city.split())
    u["normalized_city"] = city
    if errors: u["rejection_reason"] = " ".join(errors)
    return u

def check_stock(s: BakeryState) -> dict:
    return {"stock_sufficient": INITIAL_INVENTORY[s["item"]] >= s["quantity"], "steps": ["check_stock"]}

def choose_alternative(s: BakeryState) -> dict:
    options = SEQUENCE[SEQUENCE.index(s["original_item"]) + 1:][:MAX_ALTERNATIVES]
    n = s.get("alternative_attempts", 0)
    return {"item": options[n], "alternative_attempts": n + 1, "steps": ["choose_alternative"]}

def reject_order(s: BakeryState) -> dict:
    reason = s.get("rejection_reason")
    if not reason:
        n = s.get("alternative_attempts", 0)
        reason = (f"We could not fulfill the full quantity of {s['quantity']} {s['original_item']} or the {n} permitted alternative product(s)."
                  if n else f"We do not have enough {s['item']} for the full quantity of {s['quantity']}, and no alternative is available.")
    return {"status": "rejected", "delivery": "pickup", "total": Decimal(0), "rejection_reason": reason,
            "message": f"We’re sorry, but we can’t complete this order. {reason}", "steps": ["reject_order"]}

def calculate_discount(s: BakeryState) -> dict:
    subtotal = MENU_PRICES[s["item"]] * s["quantity"]
    discount = subtotal * Decimal("0.15") if subtotal > 150 and s["normalized_day"] != "friday" else Decimal(0)
    return {"subtotal": subtotal, "discount": discount, "steps": ["calculate_discount"]}

def check_delivery(s: BakeryState) -> dict:
    delivery = s["normalized_city"].casefold() == "jerusalem"
    return {"delivery": "delivery" if delivery else "pickup", "delivery_fee": Decimal(20) if delivery else Decimal(0), "steps": ["check_delivery"]}

def calculate_total(s: BakeryState) -> dict:
    return {"total": s["subtotal"] - s["discount"] + s["delivery_fee"], "status": "accepted", "steps": ["calculate_total"]}

def money(value: Decimal) -> str:
    return f"{value:.2f}".rstrip("0").rstrip(".")

def generate_message(s: BakeryState) -> dict:
    msg = f"Good news — your order for {s['quantity']} {s['item']} was accepted."
    if s["item"] != s["original_item"]:
        msg += f" We substituted {s['item']} for {s['original_item']} because the requested product was unavailable in the full quantity."
    msg += f" Your baked-goods discount is {money(s['discount'])} ILS."
    if s["delivery"] == "delivery": msg += f" We’ll deliver to Jerusalem; the fee is {money(s['delivery_fee'])} ILS."
    elif s["normalized_city"]: msg += f" We serve Jerusalem by delivery, so your order will be pickup for {s['normalized_city']}."
    else: msg += " Your order is set for pickup."
    msg += f" Your total is {money(s['total'])} ILS."
    return {"message": msg, "steps": ["generate_message"]}


## 6. Conditional Routing

The stock route checks the current item and only exposes the next alternatives within the two-attempt limit.


In [ ]:
def route_validation(s: BakeryState) -> str:
    return "reject_order" if s.get("rejection_reason") else "check_stock"

def route_stock(s: BakeryState) -> str:
    if s["stock_sufficient"]: return "calculate_discount"
    options = SEQUENCE[SEQUENCE.index(s["original_item"]) + 1:][:MAX_ALTERNATIVES]
    return "choose_alternative" if s.get("alternative_attempts", 0) < len(options) else "reject_order"


## 7. Graph Construction

Accepted orders follow the pricing and fulfillment path; rejected orders end at rejection.


In [ ]:
builder = StateGraph(BakeryState)
for name, fn in [("validate_order", validate_order), ("check_stock", check_stock),
                 ("choose_alternative", choose_alternative), ("reject_order", reject_order),
                 ("calculate_discount", calculate_discount), ("check_delivery", check_delivery),
                 ("calculate_total", calculate_total), ("generate_message", generate_message)]:
    builder.add_node(name, fn)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", route_validation, {"check_stock": "check_stock", "reject_order": "reject_order"})
builder.add_conditional_edges("check_stock", route_stock, {"calculate_discount": "calculate_discount", "choose_alternative": "choose_alternative", "reject_order": "reject_order"})
builder.add_edge("choose_alternative", "check_stock")
builder.add_edge("reject_order", END)
builder.add_edge("calculate_discount", "check_delivery")
builder.add_edge("check_delivery", "calculate_total")
builder.add_edge("calculate_total", "generate_message")
builder.add_edge("generate_message", END)
graph = builder.compile()


## 8. Graph Visualization

Display the real LangGraph diagram with its branches and substitution loop.


In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))


## 9. Test Orders

All orders below are independent and start from initial inventory.


In [ ]:
tests = [
("1 regular delivery", {"item":"challah","quantity":2,"day":"Monday","city":"Jerusalem"}, "accepted","challah","delivery",56,0),
("2 first substitute", {"item":"babka","quantity":2,"day":"Thursday","city":"Jerusalem"}, "accepted","rugelach","delivery",80,1),
("3 second substitute", {"item":"babka","quantity":4,"day":"Wednesday","city":""}, "accepted","sourdough","pickup",100,2),
("4 failed substitutes", {"item":"babka","quantity":5,"day":"Monday","city":"Jerusalem"}, "rejected","sourdough","pickup",0,2),
("5 discount", {"item":"challah","quantity":10,"day":"Thursday","city":"Jerusalem"}, "accepted","challah","delivery",173,0),
("6 Friday exclusion", {"item":"challah","quantity":10,"day":"Friday","city":"Jerusalem"}, "accepted","challah","delivery",200,0),
("7 zero quantity", {"item":"challah","quantity":0,"day":"Monday","city":"Jerusalem"}, "rejected","challah","pickup",0,0),
("8 unknown product", {"item":"pizza","quantity":2,"day":"Tuesday","city":"Jerusalem"}, "rejected","pizza","pickup",0,0),
("9 discontinued", {"item":"croissant","quantity":2,"day":"Tuesday","city":"Jerusalem"}, "rejected","croissant","pickup",0,0),
("10 outside Jerusalem", {"item":"challah","quantity":2,"day":"Tuesday","city":"Tel Aviv"}, "accepted","challah","pickup",36,0),
("11 empty-city pickup", {"item":"challah","quantity":2,"day":"Tuesday","city":""}, "accepted","challah","pickup",36,0),
("12 no stock/alternative", {"item":"challah","quantity":11,"day":"Wednesday","city":"Jerusalem"}, "rejected","challah","pickup",0,0),
("13 fractional quantity", {"item":"challah","quantity":1.5,"day":"Monday","city":"Jerusalem"}, "rejected","challah","pickup",0,0),
("14 invalid day", {"item":"challah","quantity":1,"day":"Funday","city":"Jerusalem"}, "rejected","challah","pickup",0,0),
("15 normalization", {"item":" BABKA ","quantity":2,"day":" tHuRsDaY ","city":" Jerusalem "}, "accepted","rugelach","delivery",80,1),
]


## 10. Results and Validation

Assertions compare outputs and actual node history against expected paths; execution without an exception is not considered a pass.


In [ ]:
required = {"status", "item", "delivery", "total", "message", "steps"}

def expected_steps(status, attempts, order):
    item = order.get("item")
    day = order.get("day")
    qty = order.get("quantity")
    invalid = (not isinstance(item, str) or not item.strip() or item.strip().casefold() not in MENU_PRICES or
               isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0 or
               not isinstance(day, str) or day.strip().casefold() not in WEEKDAYS)
    if status == "rejected" and invalid:
        return ["validate_order", "reject_order"]
    path = ["validate_order", "check_stock"]
    for _ in range(attempts): path += ["choose_alternative", "check_stock"]
    if status == "rejected": return path + ["reject_order"]
    return path + ["calculate_discount", "check_delivery", "calculate_total", "generate_message"]

passed_count = 0
for name, order, status, item, delivery, total, attempts in tests:
    result = graph.invoke(order)
    assert required <= result.keys(), f"{name}: missing required field"
    assert (result["status"], result["item"], result["delivery"]) == (status, item, delivery), name
    assert result["total"] == Decimal(total), name
    assert result.get("alternative_attempts", 0) == attempts <= MAX_ALTERNATIVES, name
    assert status != "rejected" or result["total"] == 0, name
    assert result["steps"] == expected_steps(status, attempts, order), (name, result["steps"])
    assert result["message"], name
    passed_count += 1
    print(f"PASS | {name} | input={order} | status={status}, item={item}, fulfillment={delivery}, total={total} ILS")
    print(f"      steps={result['steps']}")
print(f"All {passed_count}/{len(tests)} test orders passed.")
